# 2. Network layers, licensed CSPs and features of the directors

This notebook covers Sections 3.1.2 (augmenting the list of licensed CSPs), 3.1.3 (directors not providing corporate services), 3.1.4 (offshore leaks flags) and 3.2 (features, Table 1 and Appendix A.4) of the paper:

1. Read the data cleaned in notebook 1.
2. Create the network layers: companies–addresses (office, postal, PO box) and companies–directors (current and previous).
3. Find the directors that are licensed CSPs (DNB list matched by ID and name similarity).
4. Compute the features of all directors with positions in at least three independent companies.
5. Augment the list of licensed CSPs using their network, label the directors not providing corporate services, and add the remaining features.

**Output**: `data_processing/directors_clustering_extra_info.tsv`, one row per director (input of notebook 3).

Terminology used in the code: *RTO* (registered trust office) = licensed CSP; *GUO* = global ultimate owner (companies with the same GUO are not independent); `bvd_id` = Orbis company ID; `dir_id` = Orbis director ID.

In [ ]:
# Set to False to reuse the intermediate files saved by a previous run (the loops below are slow)
RECOMPUTE = True

In [ ]:
import re
import json
import pickle
from ast import literal_eval
from collections import Counter

import numpy as np
import pandas as pd
import jellyfish
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)
pd.options.mode.chained_assignment = None  # ignore SettingWithCopy warnings

## 1. Read the data

In [ ]:
# Offshore leaks: one long string per type, to search them with regular expressions
icij_adds = pickle.load(open("data_final/icij_addresses.dump", "rb+"))
icij_ents = pickle.load(open("data_final/icij_entities.dump", "rb+"))
icij_offs = pickle.load(open("data_final/icij_officers.dump", "rb+"))
icij_ints = pickle.load(open("data_final/icij_intermediaries.dump", "rb+"))

icij_adds = ":::".join([_ for _ in icij_adds if isinstance(_, str)]).upper()
icij_ents = ":::".join([_ for _ in icij_ents if isinstance(_, str)]).upper()
icij_offs = ":::".join([_ for _ in icij_offs if isinstance(_, str)]).upper()
icij_ints = ":::".join([_ for _ in icij_ints if isinstance(_, str)]).upper()

In [ ]:
# Global ultimate owner of each company
bvd2guo = json.load(open("data_final/bvd2guo.json"))

# Addresses of licensed CSPs (PO boxes are kept separately)
df_trust_addresses = pd.read_csv("data_final/df_trust_addresses.tsv", sep="\t")
df_trust_postbus = df_trust_addresses.loc[df_trust_addresses["address_full"].fillna("").str.contains("POST")]
trust_postbus = set(df_trust_postbus["address_street"])
df_trust_addresses = df_trust_addresses.loc[~df_trust_addresses["address_full"].fillna("").str.contains("POST")]

# Company information (sector, name, legal form)
df_c = pd.read_csv("data_final/df_company_info.tsv", sep="\t", converters={"nace": str})
df_c["nace"] = df_c["nace"].str.zfill(4).replace("0000", np.nan)
df_c = df_c.rename(columns={"index": "bvd_id"})
bvd2nace = df_c.set_index("bvd_id")["nace"].to_dict()
bvd2name = df_c.set_index("bvd_id")["company_name"].to_dict()
bvd_to_legal = df_c.set_index("bvd_id")["legal_form"].to_dict()

In [ ]:
def add_guo(df):
    """GUO of each company (the company itself when the owner is unknown)."""
    df["guo_id"] = df["bvd_id"].map(bvd2guo).fillna("MissingValue")
    bool_ser = df["guo_id"] == "MissingValue"
    df.loc[bool_ser, "guo_id"] = df.loc[bool_ser, "bvd_id"]
    return df

# Addresses (current and previous)
df_company_add = pd.read_csv("data_final/df_addresses.tsv", sep="\t")
df_company_add = pd.merge(df_company_add, df_c[["bvd_id", "nace"]])
df_company_add = df_company_add.fillna("")

# Branches and associations are not interesting (not matched to GUOs, same addresses and directors, noise)
df_company_add["legal_form"] = df_company_add["bvd_id"].map(bvd_to_legal)
df_company_add = df_company_add.loc[df_company_add["legal_form"] != "Branch"]
df_company_add = df_company_add.loc[df_company_add["legal_form"] != "Association"]
df_company_add = add_guo(df_company_add)

# Directors (current and previous)
df_SEOdirectors = pd.read_csv("data_final/df_directors.tsv", sep="\t")
df_SEOdirectors = add_guo(df_SEOdirectors)
df_SEOdirectors["legal_form"] = df_SEOdirectors["bvd_id"].map(bvd_to_legal)
df_SEOdirectors = df_SEOdirectors.loc[df_SEOdirectors["legal_form"] != "Branch"]
df_SEOdirectors = df_SEOdirectors.loc[df_SEOdirectors["legal_form"] != "Association"]

## 2. Network layers
### 2.1 Companies–addresses

In [ ]:
# PO box layer
df_comp_postbus = df_company_add.loc[df_company_add["address_full_norm"].str.contains(r"POSTBUS|PO\s?BOX")]
df_comp_postbus["address_postbus"] = df_company_add["address_full_norm"].copy()
df_comp_postbus_current = df_comp_postbus.loc[df_comp_postbus["address_year"] == 2019].drop_duplicates(subset=["bvd_id"])[["bvd_id", "guo_id", "address_postbus", "address_year"]]
df_comp_postbus_previous = df_comp_postbus.loc[df_comp_postbus["address_year"] < 2019][["bvd_id", "guo_id", "address_full_norm", "address_postbus", "address_year"]]

# Postal and office layers
df_comp_postal = df_company_add.loc[~df_company_add["address_full_norm"].str.contains(r"POSTBUS|PO\s?BOX")]
df_comp_office = df_comp_postal.copy()

df_comp_postal = df_comp_postal.loc[df_comp_postal["address_type"] == "Postal"]
df_comp_office = df_comp_office.loc[df_comp_office["address_type"] == "Office"]

cols = ["bvd_id", "guo_id", "address_postcode", "address_combined_gr", "address_full", "address_full_norm", "address_year"]
df_comp_postal_current = df_comp_postal.sort_values(by=["bvd_id", "address_year"], ascending=False).drop_duplicates(subset=["bvd_id"])
df_comp_office_current = df_comp_office.sort_values(by=["bvd_id", "address_year"], ascending=False).drop_duplicates(subset=["bvd_id"])

df_comp_postal_previous = df_comp_postal.loc[df_comp_postal["address_year"] < 2019, cols]
df_comp_office_previous = df_comp_office.loc[df_comp_office["address_year"] < 2019, cols]

bvd2add = df_comp_office_current.set_index("bvd_id")["address_combined_gr"].to_dict()
bvd2add_postal = df_comp_postal_current.set_index("bvd_id")["address_combined_gr"].to_dict()
bvd2add_po = df_comp_postbus_current.set_index("bvd_id")["address_postbus"].to_dict()

# Number of independent companies in each address
add_to_countoffice = df_comp_office_current.groupby("address_combined_gr")["guo_id"].agg("nunique").to_dict()
add_to_countpostal = df_comp_postal_current.groupby("address_combined_gr")["guo_id"].agg("nunique").to_dict()
add_to_countpostbus = df_comp_postbus_current.groupby("address_postbus")["guo_id"].agg("nunique").to_dict()

In [ ]:
# Licensed CSPs: Orbis IDs and addresses (all, and only those in the DNB register)
trust_ids = set(df_trust_addresses["bvd_id"].unique())
trust_add = set(df_trust_addresses["address_combined_gr"].unique()) | trust_postbus
trust_add_official = set(df_trust_addresses.loc[df_trust_addresses["official"] == 1, "address_combined_gr"].unique())

### 2.2 Companies–directors

All positions are kept (the positions held by licensed CSPs are very similar to the positions held by other directors).

In [ ]:
df_dir_current = df_SEOdirectors.loc[df_SEOdirectors["dir_status"] == "Current"]
df_dir_previous = df_SEOdirectors.loc[df_SEOdirectors["dir_status"] == "Previous"]

bvd2dir = df_dir_current.groupby(["bvd_id"])["dir_id"].agg(list).to_dict()

# Empty strings instead of missing values (to group by them and to count unknown titles and legal forms)
df_dir_current = df_dir_current.fillna("")

In [ ]:
df_comp_postal_current.to_csv("data_final/layers/df_comp_postal_current.csv", sep="\t")
df_comp_postal_previous.to_csv("data_final/layers/df_comp_postal_previous.csv", sep="\t")
df_comp_office_current.to_csv("data_final/layers/df_comp_office_current.csv", sep="\t")
df_comp_office_previous.to_csv("data_final/layers/df_comp_office_previous.csv", sep="\t")
df_comp_postbus_current.to_csv("data_final/layers/df_comp_postbus_current.csv", sep="\t")
df_comp_postbus_previous.to_csv("data_final/layers/df_comp_postbus_previous.csv", sep="\t")

df_dir_current.to_csv("data_final/layers/df_dir_current.csv", sep="\t")
df_dir_previous.to_csv("data_final/layers/df_dir_previous.csv", sep="\t")

## 3. Licensed CSPs (Section 3.1.2, step i)

Directors that are licensed CSPs: their Orbis ID (or name) is in the list of licensed CSPs, or their name is very similar to the name of a licensed CSP (cosine similarity of character 3–4-grams TF-IDF > 0.9).

In [ ]:
trust_bvd = list(df_trust_addresses["bvd_id"].dropna().unique())
trust_names = list(df_trust_addresses["company_name"].dropna().unique())

dirname2id = df_dir_current.groupby("dir_name")["dir_id"].agg(lambda x: tuple(set(x))).to_dict()
dirid2name = df_dir_current.drop_duplicates(subset=["dir_id"]).set_index("dir_id")["dir_name"].to_dict()
dirid_to_count = df_dir_current.groupby("dir_id").agg({"guo_id": "nunique"}).to_dict()["guo_id"]

dir_ids_trust = df_dir_current.loc[(df_dir_current["dir_name"].isin(trust_names)) | (df_dir_current["dir_bvd_id"].isin(trust_bvd)), "dir_id"].unique()
names_directors = list(df_dir_current["dir_name"].dropna().unique())

In [ ]:
if RECOMPUTE:
    vec_names = TfidfVectorizer(analyzer="char", ngram_range=(3, 4))
    vec_names.fit(names_directors)
    list2 = vec_names.transform(names_directors)

    # Repeat until no new names are found (names found in one iteration are used in the next one)
    len_t = 0
    while len_t != len(dir_ids_trust):
        len_t = len(dir_ids_trust)
        list1 = vec_names.transform(list(trust_names))
        cos = cosine_similarity(list1, list2, dense_output=False)

        similar_ids = []
        similar_names = []
        for j in range(cos.shape[0]):
            a = np.asarray(cos[j, :].todense()).reshape(-1)
            nonz = np.nonzero(a > 0.9)[0]
            for i in nonz:
                if a[i] < 1:  # print the matches to check them
                    print(trust_names[j], np.round(a[i], 2), np.round(jellyfish.jaro_winkler_similarity(trust_names[j], names_directors[i]), 1), names_directors[i], dirname2id[names_directors[i]])
                similar_ids += list(dirname2id[names_directors[i]])
                similar_names.append(names_directors[i])

        trust_names = list(set(trust_names) | set(similar_names))
        for i in set(similar_ids)-set(dir_ids_trust):
            print(dirid2name[i], dirid_to_count[i])
        print(len(dir_ids_trust), len(similar_ids), len(set(dir_ids_trust) | set(similar_ids)))
        dir_ids_trust = set(dir_ids_trust) | set(similar_ids)

    json.dump(list(dir_ids_trust), open("data_final/dir_ids_trust.json", "w+"))
else:
    dir_ids_trust = set(json.load(open("data_final/dir_ids_trust.json", "r+")))

In [ ]:
# Positions held by these licensed CSPs
dir_trusts = df_dir_current.loc[df_dir_current["dir_id"].isin(dir_ids_trust)]
x = dir_trusts.groupby(["dir_name"]).agg({"guo_id": "nunique", "bvd_id": "nunique"})
print(len(dir_ids_trust), "directors,", len(dir_trusts), "positions,", x.sum().to_dict())

## 4. Features of the directors (Section 3.2)

Features of the ego network (depth two) of each director: the companies they manage, the addresses, owners and other directors of those companies. Only directors with positions in at least three independent companies are kept.

In [ ]:
# Company information of each position
df_dir_current["office"] = df_dir_current["bvd_id"].map(bvd2add)
df_dir_current["postal"] = df_dir_current["bvd_id"].map(bvd2add_postal)
df_dir_current["postbus"] = df_dir_current["bvd_id"].map(bvd2add_po)
df_dir_current["nace"] = df_dir_current["bvd_id"].map(bvd2nace)
df_dir_current["company_name"] = df_dir_current["bvd_id"].map(bvd2name)
df_dir_current["dir_bvd_id"] = df_dir_current["dir_bvd_id"].fillna("")

In [ ]:
# Offshore financial centers
ofcs = ["VG", "MH", "BZ", "TW", "MT", "GI", "JE", "MU", "AI", "BM", "LU", "LR", "KY", "NR", "VC", "WS", "CY", "GY", "LI", "SC", "HK", "CW", "BS", "MC"]

def return_address_info(values, total_guo):
    """Statistics of the addresses (office, postal or PO box) of the companies of a director."""
    office = Counter(np.concatenate(values))

    office_unk = office[""]/total_guo
    del office[""]
    office = office.most_common()
    n_add = len(office)

    address_in_RTO = 0
    top_address = ""
    if len(office) > 0:
        office_top = office[0][-1]/total_guo
        top_address = office[0][0]

        address_in_RTO = sum([_[1] for _ in office if _[0] in trust_add])/sum([_[1] for _ in office])
        address_in_RTO_off = sum([_[1] for _ in office if _[0] in trust_add_official])/sum([_[1] for _ in office])
    else:
        office_top = 0
        address_in_RTO = 0
        address_in_RTO_off = 0

    return office, office_top, n_add, office_unk, address_in_RTO, address_in_RTO_off, top_address

def get_data(data2):
    """Features of a director, computed from their positions (one row per position). Returns a tab-separated string."""
    data2["office"] = data2["office"].fillna("")
    data2["postal"] = data2["postal"].fillna("")
    data2["postbus"] = data2["postbus"].fillna("")
    data2["nace"] = data2["nace"].fillna("").str[:2]
    data = data2.groupby("guo_id").agg(lambda x: list(set(x)))
    total_guo = len(data)

    office, office_top, office_n, office_unk, office_in_RTO, office_in_RTO_off, office_top_address = return_address_info(data["office"].values, total_guo)
    postal, postal_top, postal_n, postal_unk, postal_in_RTO, postal_in_RTO_off, postal_top_address = return_address_info(data["postal"].values, total_guo)
    postbus, postbus_top, postbus_n, postbus_unk, postbus_in_RTO, postbus_in_RTO_off, postbus_top_address = return_address_info(data["postbus"].values, total_guo)

    n_comps_top_office = add_to_countoffice[office_top_address] if office_top_address != "" else 0
    n_comps_top_postal = add_to_countpostal[postal_top_address] if postal_top_address != "" else 0
    n_comps_top_postbus = add_to_countpostbus[postbus_top_address] if postbus_top_address != "" else 0

    # Sectors (NACE rev. 2, 2 digits)
    nace = Counter(np.concatenate(data["nace"].values))
    nace_finance = (nace["64"]+nace["65"]+nace["66"])/total_guo
    nace_holding = nace["64"]/total_guo
    nace_real_estate = nace["68"]/total_guo
    nace_top_holding = nace["70"]/total_guo
    nace_adminis = (nace["69"] + nace["82"])/total_guo
    nace_unknown = nace[""]/total_guo
    nace_retail_wholesale = (nace["45"]+nace["46"]+nace["47"])/total_guo
    nace_construction = (nace["41"]+nace["42"]+nace["43"])/total_guo
    nace = nace.most_common()
    top_nace = nace[0][-1]/total_guo

    # Titles
    dir_title = Counter(np.concatenate(data["dir_title"].values)).most_common()
    dir_freq = sum([_[1] for _ in dir_title if re.search("Directeur|Director|Bestuurder", _[0]) is not None])/total_guo
    top_title = dir_title[0][-1]/total_guo

    # Legal forms
    legal_form = Counter(np.concatenate(data["legal_form"].values))
    legal_bv = legal_form["Private limited liability company - BV"]/total_guo
    legal_found = legal_form["Foundation"]/total_guo
    legal_vof = legal_form["General partnership - V.o.f."]/total_guo
    legal_coop = legal_form["Cooperative company - C"]/total_guo
    legal_cv = legal_form["Limited partnership - CV"]/total_guo
    legal_unk = legal_form[""]/total_guo
    legal_form = legal_form.most_common()
    top_legal = legal_form[0][-1]/total_guo

    # Owners
    iso_guo = Counter([_[:2] for _ in data.index])
    no_guo = (data2["bvd_id"] == data2["guo_id"]).sum()/total_guo
    nl_guo = (iso_guo["NL"]-no_guo*total_guo)/total_guo
    iso_guo = iso_guo.most_common()
    foreign_guo = len([_ for _ in iso_guo if _[0] != "NL"])/total_guo
    ofc_guo = len([_ for _ in iso_guo if _[0] in ofcs])/total_guo

    # Mean string similarity between the names of the companies
    cos = cosine_similarity(vec.transform(np.concatenate(data["company_name"].values)))
    mean_sim = cos[np.triu_indices(len(cos), k=1)].mean()

    vars_ = [total_guo,
             office, office_top, office_n, office_unk, office_in_RTO, office_in_RTO_off, office_top_address, n_comps_top_office,
             postal, postal_top, postal_n, postal_unk, postal_in_RTO, postal_in_RTO_off, postal_top_address, n_comps_top_postal,
             postbus, postbus_top, postbus_n, postbus_unk, postbus_in_RTO, postbus_in_RTO_off, postbus_top_address, n_comps_top_postbus,
             nace, nace_finance, nace_holding, nace_real_estate, nace_top_holding, nace_adminis, nace_unknown, nace_retail_wholesale, nace_construction, nace, top_nace,
             dir_title, dir_freq, top_title,
             legal_form, legal_bv, legal_found, legal_vof, legal_coop, legal_cv, legal_unk, top_legal,
             iso_guo, no_guo, nl_guo, foreign_guo, ofc_guo, mean_sim]
    return ("\t{}"*len(vars_)).format(*vars_)

In [ ]:
# Character n-gram TF-IDF of the company names (string similarity features)
vec = TfidfVectorizer(analyzer="char", ngram_range=(3, 4))
vec.fit(list(df_c["company_name"]))

group_dirs = df_dir_current.groupby(["dir_id", "dir_name", "dir_bvd_id"])

if RECOMPUTE:
    with open("data_processing/directors_clustering.tsv", "w+") as f:
        for dir_, data2 in tqdm(group_dirs):
            # Only directors with positions in at least 3 independent companies
            if len(data2["guo_id"].unique()) > 2:
                flag = dir_[0] in dir_ids_trust
                total_cs = len(data2)
                comps = set(data2["bvd_id"])
                vars_str = get_data(data2)

                # Other directors of the companies
                x = df_dir_current.loc[df_dir_current["bvd_id"].isin(comps)]
                x = x.groupby("guo_id")["dir_id"].agg(lambda x: list(set(x)))
                av_directors_comps = x.str.len().mean()  # average number of directors per independent company

                x = Counter(np.concatenate(x.values)).most_common()
                shared_directors = [_ for _ in x if (_[1] > 1) and (_[0] != dir_[0])]
                number_shared_directors = sum([_[1] for _ in shared_directors])
                RTO_shared_directors = sum([_[1] for _ in x if _[0] in dir_ids_trust])

                f.write("{}\t{}\t{}\t{}\t{}\t{}\t{}\t{}\t{}\t{}\t{}{}\n".format(*dir_, flag, total_cs, av_directors_comps, shared_directors,
                                                                                number_shared_directors, RTO_shared_directors, comps, x, vars_str))

In [ ]:
df_cluster = pd.read_csv("data_processing/directors_clustering.tsv", sep="\t", header=None)
df_cluster.columns = ["dir_id", "dir_name", "dir_bvd_id", "RTO", "n_comp", "av_directors_comps", "shared_directors", "number_shared_directors", "RTO_shared_directors", "comps", "dir_ids",
                      "total_guo", "office", "office_top", "office_n", "office_unk", "office_in_RTO", "office_in_RTO_off", "office_top_address", "n_comps_top_office",
                      "postal", "postal_top", "postal_n", "postal_unk", "postal_in_RTO", "postal_in_RTO_off", "postal_top_address", "n_comps_top_postal",
                      "postbus", "postbus_top", "postbus_n", "postbus_unk", "postbus_in_RTO", "postbus_in_RTO_off", "postbus_top_address", "n_comps_top_postbus",
                      "nace", "nace_finance", "nace_holding", "nace_real_estate", "nace_top_holding", "nace_adminis", "nace_unknown", "nace_retail_wholesale", "nace_construction", "nace", "top_nace",
                      "dir_title", "dir_freq", "top_title",
                      "legal_form", "legal_bv", "legal_found", "legal_vof", "legal_coop", "legal_cv", "legal_unk", "top_legal",
                      "iso_guo", "no_guo", "nl_guo", "foreign_guo", "ofc_guo", "mean_sim"]
df_cluster["av_companies_guo"] = df_cluster["n_comp"]/df_cluster["total_guo"]
len(df_cluster)

In [ ]:
# Corporate directors: do they belong to the same group as the companies they manage?
dir_2_bvd = df_dir_current.dropna(subset=["dir_bvd_id"]).set_index("dir_id")["dir_bvd_id"].to_dict()

df_cluster["dir_ids"] = df_cluster["dir_ids"].apply(literal_eval)
df_cluster["comps"] = df_cluster["comps"].apply(literal_eval)
df_cluster["guos"] = df_cluster["comps"].apply(lambda x: [bvd2guo[_] if bvd2guo.get(_) is not None else _ for _ in x])

df_cluster["dir_id_bvd"] = df_cluster["dir_id"].map(dir_2_bvd)
df_cluster["dir_id_bvd_guo"] = df_cluster["dir_id_bvd"].map(bvd2guo)
df_cluster["dir_id_bvds"] = df_cluster[["dir_id_bvd_guo", "dir_id_bvd"]].apply(lambda x: set(x)-{np.nan}, axis=1)

df_cluster["same_guo"] = df_cluster[["dir_id_bvds", "guos"]].apply(lambda x: len([_ for _ in x[0] if _ in x[1]]), axis=1)
df_cluster["same_guo_frac"] = df_cluster["same_guo"]/df_cluster["n_comp"]

In [ ]:
def get_similarity_dir_comps(x):
    """Mean string similarity between the name of the director and the names of the companies."""
    trans = vec.transform([x[0]] + [bvd2name[_] for _ in x[1]])
    cos = cosine_similarity(trans[0], trans[1:]).mean()
    return cos.mean()

df_cluster["mean_sim_dir_comps"] = df_cluster[["dir_name", "comps"]].apply(get_similarity_dir_comps, axis=1)
df_cluster["RTO_shared_directors_perc"] = df_cluster["RTO_shared_directors"]/df_cluster["total_guo"]

## 5. Augmented list of licensed CSPs (Section 3.1.2, step iib)

Directors with over 25% of their companies in the address of a licensed CSP (DNB list) and over 20% of their companies with another director from a licensed CSP; or with over 50% of their companies in the address of a licensed CSP (augmented list) and over 50% of their companies with another director from a licensed CSP.

In [ ]:
crit1 = set(df_cluster.loc[((df_cluster["office_in_RTO_off"] > 0.25) | (df_cluster["postal_in_RTO_off"] > 0.25)) & (df_cluster["RTO_shared_directors_perc"] > 0.20), "dir_id"])
crit2 = set(df_cluster.loc[((df_cluster["office_in_RTO"] > 0.5) | (df_cluster["postal_in_RTO"] > 0.5)) & (df_cluster["RTO_shared_directors_perc"] > 0.5), "dir_id"])
crit3 = set(df_cluster.loc[(df_cluster["RTO_shared_directors_perc"] > 0.2) & (df_cluster["RTO_shared_directors"] > 10), "dir_id"])

# New directors found by each criterion, excluding the directors holding a position in a licensed CSP (step iia)
dir_ids_trust_c = set(dir_ids_trust)
pos_in_rto = set(df_dir_current.loc[df_dir_current["bvd_id"].isin(trust_ids), "dir_id"])
print(len((crit1 | crit2 | crit3) - dir_ids_trust_c), "new directors;",
      len(crit1 - dir_ids_trust_c - pos_in_rto), len(crit2 - dir_ids_trust_c - crit1 - pos_in_rto), len(crit3 - dir_ids_trust_c - crit1 - crit2 - pos_in_rto),
      "excluding directors of licensed CSPs")

dir_ids_trust_c = dir_ids_trust_c | crit1 | crit2 | crit3
json.dump(list(dir_ids_trust_c), open("data_final/dir_ids_trust_c.json", "w+"))

In [ ]:
df_cluster["total_guo"] = df_cluster["comps"].apply(lambda x: len(set([bvd2guo[_] if bvd2guo.get(_) is not None else _ for _ in x])))
df_cluster = df_cluster.reset_index(drop=True)

Flag the companies (current and previous positions) managed by each type of licensed CSP:
RTO7 = step i (DNB list), RTO6 = directors holding a position in a licensed CSP (step iia), RTO4 = augmented list (step iib).

In [ ]:
dir_2_num_pos_rto = df_dir_current.loc[df_dir_current["bvd_id"].isin(trust_ids), "dir_id"].value_counts().to_dict()

df_dir_current["dir_id_RTO7"] = df_dir_current["dir_id"].isin(dir_ids_trust)
df_dir_current["dir_id_RTO6"] = df_dir_current["dir_id"].isin(dir_2_num_pos_rto)
df_dir_current["dir_id_RTO4"] = df_dir_current["dir_id"].isin(dir_ids_trust_c-dir_ids_trust)

bvd_2_flag_rto = df_dir_current.groupby("bvd_id")[["dir_id_RTO7", "dir_id_RTO6", "dir_id_RTO4"]].sum().to_dict()
json.dump(bvd_2_flag_rto, open("data_processing/bvd_2_flag_rto.json", "w+"))

df_cluster["N_pos_RTO"] = df_cluster["dir_id"].map(dir_2_num_pos_rto).fillna(0)

In [ ]:
# Previous directors and addresses
df_dir_previous["dir_id_RTO7"] = df_dir_previous["dir_id"].isin(dir_ids_trust) | df_dir_current["dir_name"].isin(trust_names)
df_dir_previous["dir_id_RTO6"] = df_dir_previous["dir_id"].isin(dir_2_num_pos_rto)
df_dir_previous["dir_id_RTO4"] = df_dir_previous["dir_id"].isin(dir_ids_trust_c-dir_ids_trust)
df_dir_previous["c"] = 1

bvd2prev_dir_rto7 = set(df_dir_previous.loc[df_dir_previous["dir_id_RTO7"] == True, "bvd_id"].unique())
bvd2prev_dir_rto6 = set(df_dir_previous.loc[df_dir_previous["dir_id_RTO6"] == True, "bvd_id"].unique())
bvd2prev_dir_rto4 = set(df_dir_previous.loc[df_dir_previous["dir_id_RTO4"] == True, "bvd_id"].unique())

before_in_trust = set(df_comp_office_previous.loc[df_comp_office_previous["address_combined_gr"].isin(trust_add), "bvd_id"].unique())
before_in_trust_off = set(df_comp_office_previous.loc[df_comp_office_previous["address_combined_gr"].isin(trust_add_official), "bvd_id"].unique())

pickle.dump(bvd2prev_dir_rto7, open("data_final/bvd2prev_dir_rto7.pkl", "wb+"))
pickle.dump(bvd2prev_dir_rto6, open("data_final/bvd2prev_dir_rto6.pkl", "wb+"))
pickle.dump(bvd2prev_dir_rto4, open("data_final/bvd2prev_dir_rto4.pkl", "wb+"))
pickle.dump(before_in_trust, open("data_final/before_in_trust.pkl", "wb+"))
pickle.dump(before_in_trust_off, open("data_final/before_in_trust_off.pkl", "wb+"))

In [ ]:
bvd2dir_prev = df_dir_previous.groupby("bvd_id")["c"].sum().to_dict()
bvd2add_prev = df_comp_office_previous.groupby("bvd_id")["address_year"].count().to_dict()

df_cluster["n_add_prev_trust"] = df_cluster["comps"].apply(lambda x: len([_ for _ in x if _ in before_in_trust]))
df_cluster["n_add_prev_trust_off"] = df_cluster["comps"].apply(lambda x: len([_ for _ in x if _ in before_in_trust_off]))
df_cluster["frac_add_prev_trust"] = df_cluster["n_add_prev_trust"]/df_cluster["n_comp"]
df_cluster["frac_add_prev_trust_off"] = df_cluster["n_add_prev_trust_off"]/df_cluster["n_comp"]

df_cluster["n_dir_prev_rto7"] = df_cluster["comps"].apply(lambda x: len([_ for _ in x if _ in bvd2prev_dir_rto7]))
df_cluster["n_dir_prev_rto6"] = df_cluster["comps"].apply(lambda x: len([_ for _ in x if _ in bvd2prev_dir_rto6]))
df_cluster["n_dir_prev_rto4"] = df_cluster["comps"].apply(lambda x: len([_ for _ in x if _ in bvd2prev_dir_rto4]))
df_cluster["frac_dir_prev_rto7"] = df_cluster["n_dir_prev_rto7"]/df_cluster["n_comp"]
df_cluster["frac_dir_prev_rto6"] = df_cluster["n_dir_prev_rto6"]/df_cluster["n_comp"]
df_cluster["frac_dir_prev_rto4"] = df_cluster["n_dir_prev_rto4"]/df_cluster["n_comp"]

df_cluster["av_n_prev_dir"] = df_cluster["comps"].apply(lambda x: np.mean([bvd2dir_prev[_] for _ in x if _ in bvd2dir_prev])).fillna(0)
df_cluster["av_n_prev_add"] = df_cluster["comps"].apply(lambda x: np.mean([bvd2add_prev[_] for _ in x if _ in bvd2add_prev])).fillna(0)

### Labels

`RTO_confidence`: 7 = licensed CSP from the DNB list (step i), 6 = holds a position in a licensed CSP (step iia), 4 = augmented list (step iib), 1 = other directors, 0 = directors not providing corporate services (Section 3.1.3: corporate directors whose own sector is not in NACE 64–66, 69, 70 or 82).

In [ ]:
df_cluster["RTO_ext"] = df_cluster["dir_id"].isin(dir_ids_trust_c)
df_cluster["possible_RTO"] = (df_cluster["office_in_RTO"] > 0.5) | (df_cluster["postal_in_RTO"] > 0.5)
df_cluster["RTO_confidence"] = 100 + df_cluster["RTO"].astype(int)*10 + df_cluster["RTO_ext"].astype(int)*1
df_cluster["RTO_confidence"] = df_cluster["RTO_confidence"].replace({100: 1, 101: 4, 111: 7})
df_cluster.loc[(df_cluster["N_pos_RTO"] > 0) & (df_cluster["RTO_confidence"] < 7), "RTO_confidence"] = 6

no_trust = df_c.loc[~df_c["nace"].str[:2].isin(["64", "65", "66", "69", "70", "82"]), "bvd_id"]
no_trust_dir = df_dir_current.loc[df_dir_current["dir_bvd_id"].isin(no_trust), "dir_id"]
df_cluster["Not_RTO"] = df_cluster["dir_id"].isin(set(no_trust_dir))
df_cluster.loc[df_cluster["Not_RTO"], "RTO_confidence"] = 0

print(df_cluster["RTO_confidence"].value_counts())
# Positions held by the licensed CSPs of each step
print(df_cluster.groupby("RTO_confidence")["n_comp"].sum())

### Remaining features

In [ ]:
# Corporate directors: number of directors shared with the companies they manage
df_cluster["same_directors"] = df_cluster[["dir_id_bvd", "dir_id_bvd_guo", "comps"]].apply(lambda x: {c for _ in x[2] for c in bvd2dir[_]} &
                                                                 (set(bvd2dir.get(x[0]) if bvd2dir.get(x[0]) is not None else []) |
                                                                  set(bvd2dir.get(x[1]) if bvd2dir.get(x[1]) is not None else [])),
                                                                 axis=1).str.len()

df_cluster["log_total_guo"] = np.log10(df_cluster["total_guo"])
df_cluster["log_n_comp"] = np.log10(df_cluster["n_comp"])
df_cluster["log_n_comps_top_postbus"] = np.log10(1+df_cluster["n_comps_top_postbus"])
df_cluster["log_n_comps_top_postal"] = np.log10(1+df_cluster["n_comps_top_postal"])
df_cluster["log_n_comps_top_office"] = np.log10(1+df_cluster["n_comps_top_office"])

df_cluster["office"] = df_cluster["office"].apply(literal_eval)
df_cluster["postal"] = df_cluster["postal"].apply(literal_eval)
df_cluster["company_names"] = df_cluster["comps"].apply(lambda x: [bvd2name.get(_) for _ in x])

df_cluster["corp"] = df_cluster["dir_id"].str[0] == "C"  # corporate director (vs individual)

In [ ]:
# Offshore leaks (Section 3.1.4)
remove_titles = set(['MR', 'MS', 'DRS.', 'IR.', 'DR.', 'DR', 'PROF.', 'ING.', 'MD', 'DRS', 'BARON', 'ING', 'GRAAF', 'MAG.', 'PROFESSOR', 'DIPL.-KFM.', 'IR', 'DIPL.-ING.'])

def query_icij(q, which="dir"):
    """Number of times a postcode (which="add"), a director name ("dir") or a company name (other) appears in the offshore leaks."""
    if q is None:
        return 0
    if which == "add":
        q = q.split("|")[0].strip()
        if len(q) != 6:
            return 0
        else:
            return len([1 for _ in re.finditer(q, icij_adds)])
    elif which == "dir":
        if q.split()[0] in remove_titles:
            q = " ".join(q.split()[1:])
    q = re.escape(q)
    return len([1 for _ in re.finditer(q, icij_ents)]) +\
        len([1 for _ in re.finditer(q, icij_offs)]) +\
        len([1 for _ in re.finditer(q, icij_ints)])

df_cluster["n_add_off_icij"] = df_cluster["office"].apply(lambda x: sum([_[1]*query_icij(_[0], which="add") for _ in x]))
df_cluster["n_add_post_icij"] = df_cluster["postal"].apply(lambda x: sum([_[1]*query_icij(_[0], which="add") for _ in x]))
df_cluster["n_comp_icij"] = df_cluster["company_names"].apply(lambda x: sum([query_icij(_, which="comp") for _ in x]))
df_cluster["n_dir_icij"] = df_cluster["dir_name"].apply(query_icij, which="dir")
df_cluster["frac_add_off_icij"] = df_cluster["n_add_off_icij"]/df_cluster["n_comp"]
df_cluster["frac_add_post_icij"] = df_cluster["n_add_post_icij"]/df_cluster["n_comp"]
df_cluster["frac_comp_icij"] = df_cluster["n_comp_icij"]/df_cluster["n_comp"]

In [ ]:
df_cluster["dir_nace"] = df_cluster["dir_id_bvd"].map(bvd2nace)

df_cluster.to_csv("data_processing/directors_clustering_extra_info.tsv", sep="\t", index=False)